In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
sc.settings.set_figure_params(dpi=150, facecolor="white")
sc.settings.figdir = r"Z:\Bel\Marina_Side_Projects\scRNA\Figures"

## HF_OLD_YOUNG

### Read data

In [ ]:
results_file = '/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/hf_old_young_2.h5ad'

In [ ]:
#Read data
hf_old = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_059/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
hf_young = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_060/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
hf_old_young = sc.concat([hf_old, hf_young], axis=0, label='cell_age', keys= ['hf_old', 'hf_young'], index_unique='_', join='outer')
hf_old_young.obs_names_make_unique()

### Preprocessing

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
hf_old_young.var["mt"] = hf_old_young.var_names.str.startswith("mt-")
# ribosomal genes
hf_old_young.var["ribo"] = hf_old_young.var_names.str.startswith(("Rps", "Rpl"))
# dissociation genes
diss_genes = ["Atf3", "Btg2", "Dusp1", "Egr1", "Fos", "Fosb", "Ier2", "Ifrd1", "Junb", "Nr4a1", "Zfp36"]
hf_old_young.var['diss'] = hf_old_young.var_names.isin(diss_genes)
# calculate qc metrics
sc.pp.calculate_qc_metrics(hf_old_young, qc_vars=["mt", "ribo", "diss"], inplace=True, log1p=True)

In [ ]:
#Plot qc metrics_violin
sc.pl.violin(hf_old_young,["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_diss"], jitter=0.4, multi_panel=True)

In [ ]:
#Plot qc metrics_scatter
sc.pl.scatter(hf_old_young, 'total_counts', 'n_genes_by_counts', color='pct_counts_mt')

In [ ]:
#Filter cells
sc.pp.filter_cells(hf_old_young, min_genes=100)
sc.pp.filter_genes(hf_old_young, min_cells=3)
hf_old_young = hf_old_young[hf_old_young.obs.n_genes_by_counts < 6500, :]
hf_old_young = hf_old_young[hf_old_young.obs.pct_counts_mt < 20, :].copy()
hf_old_young = hf_old_young[hf_old_young.obs.pct_counts_ribo < 25, :].copy()
hf_old_young = hf_old_young[hf_old_young.obs.pct_counts_diss < 1, :].copy()

In [ ]:
#Doublets detection
sc.pp.scrublet(hf_old_young)

In [ ]:
hf_old_young.obs

In [ ]:
# Saving count data
hf_old_young.layers["counts"] = hf_old_young.X.copy()

In [ ]:
hf_old_young.raw = sc.pp.log1p(hf_old_young, copy = True)

### Data normalization and scale

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(hf_old_young)
# Logarithmize the data
sc.pp.log1p(hf_old_young)

In [ ]:
#Highly variable selection
sc.pp.highly_variable_genes(hf_old_young, n_top_genes=2000, batch_key='cell_age')

In [ ]:
#Cell cycle scoring
cell_cycle_genes = [x.strip().title() for x in open('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/regev_lab_cell_cycle_genes.txt')]
# Separate dataset for S-phase or G2M-phase associated genes
s_genes = [x for x in cell_cycle_genes[:43]]
g2m_genes = [x for x in cell_cycle_genes[43:]]
sc.tl.score_genes_cell_cycle(hf_old_young, s_genes = s_genes, g2m_genes=g2m_genes)

In [ ]:
hf_old_young.obs['CC_Difference'] = hf_old_young.obs[str('S_score')] - hf_old_young.obs[str('G2M_score')]

In [ ]:
sc.pl.highly_variable_genes(hf_old_young)

In [ ]:
hf_old_young.obs

In [ ]:
sc.pp.regress_out(hf_old_young, ['CC_Difference', 'pct_counts_mt', 'pct_counts_ribo'])

In [ ]:
sc.pp.scale(hf_old_young, max_value=10)

### PCA, UMAP and Clustering

In [ ]:
#PCA
sc.tl.pca(hf_old_young)

In [ ]:
sc.pl.pca_variance_ratio(hf_old_young, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    hf_old_young,
    color=["cell_age", "cell_age", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

In [ ]:
#Nearest neighbor
sc.pp.neighbors(hf_old_young, n_neighbors=30)
#Umap
sc.tl.umap(hf_old_young, min_dist=1, spread=2)

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster, especially for larger datasets
sc.tl.leiden(hf_old_young, flavor="igraph", n_iterations=-1, resolution = 0.5)

In [ ]:
# Read directly to just plot
hf_old_young = sc.read_h5ad('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Bulge_3_5.h5ad')

In [ ]:
sc.pl.umap(hf_old_young, color=["leiden"], show=False)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/hf_old_young_umap.tif', format='tiff', bbox_inches='tight', dpi=300 )

In [ ]:
expression_matrix = hf_old_young.raw.to_adata().to_df().T if hf_old_young.raw else hf_old_young.to_df().T

In [ ]:
expression_matrix.to_csv("/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/bulge_3_5_gene_expression.csv")

In [ ]:
sc.pl.umap(
    hf_old_young,
    color=["leiden", "doublet_score", "predicted_doublet"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
)

In [ ]:
sc.pl.umap(
    hf_old_young,
    color=["leiden", "Lgr5", "Cd34", 'Krt14', 'pct_counts_diss', 'cell_age'],
    wspace=0.5,
    ncols=3,
)

In [ ]:
fig, axes = plt.subplots(ncols = 2, nrows=1, figsize = (10, 4), gridspec_kw={'wspace': 0.5})
size = 10
wspace = 0.5
gene = 'leiden'
ax = axes[0]
sc.pl.umap(hf_old_young[hf_old_young.obs['cell_age']!='hf_mid'], show = False, ax = ax, size = size)
sc.pl.umap(hf_old_young[hf_old_young.obs['cell_age']=='hf_old'], color= gene,  size = size, title= 'Old peroxisomes', wspace = wspace, ax = ax, show = False)
ax = axes[1]
sc.pl.umap(hf_old_young[hf_old_young.obs['cell_age']!='hf_mid'], show = False, ax = ax, size = size)
sc.pl.umap(hf_old_young[hf_old_young.obs['cell_age']=='hf_young'], color= gene,  size = size, title= 'Young peroxisomes', wspace = wspace, ax = ax, show = False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/HF_umap_orig.tif', format='tiff',bbox_inches='tight', dpi = 300)

In [ ]:
hf_old_young.obs['X_umap_x'] = hf_old_young.obsm['X_umap'][:,0]
hf_old_young.obs['X_umap_y'] =  hf_old_young.obsm['X_umap'][:,1]

In [ ]:
hf_old_young.obs['test'] = ((hf_old_young.obs['cell type'] == 'Bulge') & 
                            (hf_old_young.obs['X_umap_x'] < 0.5) &  
                            (hf_old_young.obs['X_umap_y'] > 0.75))

In [ ]:
hf_old_young.obs

### Cluster annotation

In [ ]:
sc.pl.umap(
    hf_old_young,
    color=["Lgr5", "Cd34", "Sox9", "Lgr6", "Lrig1", "Krt10", "Cidea", "Mgst1", "Cd200", "leiden"],
    wspace=0.5,
    ncols=3,
    cmap='viridis',
    show=False
)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/HF_Umap_Markers.tif', format='tiff',bbox_inches='tight', dpi = 300)

In [ ]:
#Identification of clusters based on known markers

marker_genes = {
    'HF_Suprabasal':['Krt79', 'Krt10'],
    'UpHF': ['Lgr6', 'Lrig1'],
    'Bulge': ['Lgr5', 'Cd34', 'Krt15'],
    'SG': ['Cidea', 'Mgst1'],
    'Basal': ['Krt14']
}
plt.figure()
sc.pl.dotplot(hf_old_young, marker_genes, 'leiden', dendrogram=False, show=False)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Cluster_annotation.tif', format='tiff', bbox_inches='tight', dpi=300)
#plt.close()

In [ ]:
#Create a dictionary to map clusters to annotation labels

Cluster2annotation = {
    '0': 'HF_Suprabasal_1',  
    '1': 'Isthmus_SC', 
    '2': 'Bulge',
    '3': 'Isthmus_1',
    '4': 'Isthmus_2',
    '5': 'HF_Suprabasal_2',
    '6': 'SG',
    '7': 'Misc',
    '8': 'Infundibulum'
}
#Add annotations to obs
hf_old_young.obs["cell type"] = hf_old_young.obs["leiden"].map(Cluster2annotation).astype("category")
sc.pl.dotplot(hf_old_young, marker_genes, 'cell type', dendrogram=False)

In [ ]:
ax = sc.pl.umap(
    hf_old_young,
    color="cell type",
    legend_loc="on data",
    frameon=False,
    legend_fontsize=5, size=10,
    show = False, na_in_legend = False)
sc.pl.umap(hf_old_young, color="cell type",
    frameon=False,
    legend_fontsize=5, size = 0, ax = ax, na_in_legend = False, show = False)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/HF_Cluster_annotation.tif', format='tiff', bbox_inches='tight', dpi=300 )

In [ ]:
hf_old_young.write(results_file)

### DE expressed genes

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(hf_old_young, groupby="leiden", method="wilcoxon",groups=['3', '4'])

In [ ]:
old_pero_bulge_genes = sc.get.rank_genes_groups_df(hf_old_young, group="3")
#old_pero_bulge_genes.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/old_pero_bulge_genes.csv', index=False)

In [ ]:
top_15_DE_genes

In [ ]:
#Volcano Plot
DE_genes = old_pero_bulge_genes
#Add -log10(pval) column
DE_genes['-log10(pval)'] = -np.log10(DE_genes['pvals'])
#logfoldchange threshold
logFC_threshold = 12
DE_genes = DE_genes[abs(DE_genes['logfoldchanges']) <= logFC_threshold]
# Define thresholds for significance
log2FoldChange_threshold = 1  
pval_threshold = 0.05 
s = 10
# Determine significance
DE_genes['significant'] = (abs(DE_genes['logfoldchanges']) > log2FoldChange_threshold) & (DE_genes['pvals'] < pval_threshold)
# Get top 15 significant genes sorted by p-value (or you could sort by log2 fold change if desired)
top_15_DE_genes = DE_genes[DE_genes['significant']].nsmallest(25, 'pvals')
# Plotting
plt.figure(figsize=(5, 4))
# Scatter plot for non-significant genes
plt.scatter(DE_genes['logfoldchanges'], DE_genes['-log10(pval)'], c='grey', alpha=0.7, s=s)
# Scatter plot for significant genes (color coding by sign of logfoldchange)
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), '-log10(pval)'], 
            c='red', alpha=0.7, s=s, label='Upregulated')
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), '-log10(pval)'], 
            c='blue', alpha=0.7, s=s, label='Downregulated')
# Label significant genes
# Label the top 15 significant genes
for i, row in top_15_DE_genes.iterrows():
    plt.text(row['logfoldchanges'], row['-log10(pval)'], row['names'], fontsize=8, ha='right')
# Adding labels and title
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 p-value')
plt.title('Bulge Old vs Young')
plt.axhline(-np.log10(pval_threshold), color='green', linestyle='--')  # Add a horizontal line for p-value threshold
plt.axvline(log2FoldChange_threshold, color='green', linestyle='--')   # Add vertical lines for log2 fold change threshold
plt.axvline(-log2FoldChange_threshold, color='green', linestyle='--')
plt.grid(False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/Bulge_Old_Young.tif', format='tiff', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sc.pl.violin(Bulge, keys = ['Ly6d','Cst6', 'Defb6', 'Krt79'], groupby = 'cell_age')
sc.pl.violin(Bulge, keys = ['Lsmem1','Ldb2', 'Slc7a8', 'Ier2'], groupby = 'cell_age')
sc.pl.violin(Bulge, keys = ['Fam71a','Krt77', 'Hoxd8', 'Ly6g6c'], groupby = 'cell_age')
sc.pl.violin(Bulge, keys = ['Sprr1a','Klf2', 'Skint3', 'Ppm1n'], groupby = 'cell_age')

In [ ]:
UpHF_SC = hf_old_young[hf_old_young.obs.leiden=='2'].copy()
UpHF_SC.X = UpHF_SC.layers['counts'].toarray().copy()

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(UpHF_SC, groupby="cell_age", method="wilcoxon")

In [ ]:
old_pero_UpHF_genes = sc.get.rank_genes_groups_df(UpHF_SC, group="hf_old")
old_pero_UpHF_genes.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/old_pero_UpHF_genes.csv', index=False)

In [ ]:
top_15_DE_genes

In [ ]:
sc.pl.violin(hf_old_young, keys = ['Klf2','Bdnf'], groupby = 'cell type')
sc.pl.violin(hf_old_young, keys = ['Plk3','Scd1'], groupby = 'cell type')
sc.pl.violin(hf_old_young, keys = ['Atg9a', 'Prkd1'], groupby = 'cell type')

In [ ]:
#Volcano Plot
DE_genes = old_pero_UpHF_genes
#Add -log10(pval) column
DE_genes['-log10(pval)'] = -np.log10(DE_genes['pvals'])
#logfoldchange threshold
logFC_threshold = 10
DE_genes = DE_genes[abs(DE_genes['logfoldchanges']) <= logFC_threshold]
# Define thresholds for significance
log2FoldChange_threshold = 1  
pval_threshold = 0.05 
s = 10
# Determine significance
DE_genes['significant'] = (abs(DE_genes['logfoldchanges']) > log2FoldChange_threshold) & (DE_genes['pvals'] < pval_threshold)
# Get top 15 significant genes sorted by p-value (or you could sort by log2 fold change if desired)
top_15_DE_genes = DE_genes[DE_genes['significant']].nsmallest(50, 'pvals')
# Plotting
plt.figure(figsize=(5, 4))
# Scatter plot for non-significant genes
plt.scatter(DE_genes['logfoldchanges'], DE_genes['-log10(pval)'], c='grey', alpha=0.7, s=s)
# Scatter plot for significant genes (color coding by sign of logfoldchange)
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), '-log10(pval)'], 
            c='red', alpha=0.7, s=s, label='Upregulated')
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), '-log10(pval)'], 
            c='blue', alpha=0.7, s=s, label='Downregulated')
# Label significant genes
# Label the top 15 significant genes
for i, row in top_15_DE_genes.iterrows():
    plt.text(row['logfoldchanges'], row['-log10(pval)'], row['names'], fontsize=8, ha='right')
# Adding labels and title
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 p-value')
plt.title('Subcluster 0 Old DEGs')
plt.axhline(-np.log10(pval_threshold), color='green', linestyle='--')  # Add a horizontal line for p-value threshold
plt.axvline(log2FoldChange_threshold, color='green', linestyle='--')   # Add vertical lines for log2 fold change threshold
plt.axvline(-log2FoldChange_threshold, color='green', linestyle='--')
plt.grid(False)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/Subcluster_0_DEG.tif', format='tiff', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
old_pero_de_genes.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/old_pero_de_genes.csv', index=False)

In [ ]:
hf_old_young = sc.read_h5ad('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/hf_old_young.h5ad')

In [ ]:
non_nan_mask = pd.notna(hf_old_young.obs['cell type'])
hf_old_young = hf_old_young[non_nan_mask]

In [ ]:
hf_old_young.obs

In [ ]:
import decoupler as dc

In [ ]:
LogFCs, pvals = dc.get_contrast(
    hf_old_young,
    group_col="cell type",
    condition_col="cell_age",
    condition="hf_old",
    reference="hf_young",
    method="wilcoxon",
)

In [ ]:
LogFCs, pvals

In [ ]:
fig, axes = plt.subplots(ncols = 2, nrows=2, figsize = (10, 10), gridspec_kw={'wspace': 0.5, 'hspace': 0.4})
ax = axes[0, 0]
dc.plot_volcano(LogFCs, pvals, 'Bulge', top=15, sign_thr=0.05, lFCs_thr=1, lFCs_limit=10, ax=ax)
ax.grid(False)
ax = axes[0, 1]
dc.plot_volcano(LogFCs, pvals, 'Up_HF', top=15, sign_thr=0.05, lFCs_thr=1, lFCs_limit=10, ax=ax)
ax.grid(False)
ax = axes[1, 0]
dc.plot_volcano(LogFCs, pvals, 'HF_Suprabasal', top=15, sign_thr=0.05, lFCs_thr=1, lFCs_limit=10, ax=ax)
ax.grid(False)
ax = axes[1, 1]
dc.plot_volcano(LogFCs, pvals, 'SG', top=15, sign_thr=0.05, lFCs_thr=1, lFCs_limit=10, ax=ax)
ax.grid(False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/HF_DEG.tif', format='tiff',bbox_inches='tight', dpi = 300)

## IFE_OLD_YOUNG

### Read data

In [ ]:
results_file = '/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/ife_old_young.h5ad'

In [ ]:
#Read data
ife_old = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_056/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
ife_young = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_057/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
#Merge data
ife_old_young = sc.concat([ife_old, ife_young], axis=0, label='cell_age', keys= ['ife_old', 'ife_young'], index_unique='_', join='outer')
ife_old_young.obs_names_make_unique()

### Preprocessing

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
ife_old_young.var["mt"] = ife_old_young.var_names.str.startswith("mt-")
# ribosomal genes
ife_old_young.var["ribo"] = ife_old_young.var_names.str.startswith(("Rps", "Rpl"))
# dissociation genes
diss_genes = ["Atf3", "Btg2", "Dusp1", "Egr1", "Fos", "Fosb", "Ier2", "Ifrd1", "Junb", "Nr4a1", "Zfp36"]
ife_old_young.var['diss'] = ife_old_young.var_names.isin(diss_genes)
# calculate qc metrics
sc.pp.calculate_qc_metrics(ife_old_young, qc_vars=["mt", "ribo", "diss"], inplace=True, log1p=True)

In [ ]:
#Plot qc metrics_violin
sc.pl.violin(ife_old_young,["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_diss"], jitter=0.4, multi_panel=True)

In [ ]:
#Plot qc metrics_scatter
sc.pl.scatter(ife_old_young, 'total_counts', 'n_genes_by_counts', color='pct_counts_mt')

In [ ]:
#Filter cells
sc.pp.filter_cells(ife_old_young, min_genes=100)
sc.pp.filter_genes(ife_old_young, min_cells=3)
ife_old_young = ife_old_young[ife_old_young.obs.n_genes_by_counts < 6500, :]
ife_old_young = ife_old_young[ife_old_young.obs.pct_counts_mt < 20, :].copy()
ife_old_young = ife_old_young[ife_old_young.obs.pct_counts_ribo < 30, :].copy()
ife_old_young = ife_old_young[ife_old_young.obs.pct_counts_diss < 2.5, :].copy()

In [ ]:
#Doublets detection
sc.pp.scrublet(ife_old_young)

In [ ]:
ife_old_young.obs

In [ ]:
# Saving count data
ife_old_young.layers["counts"] = ife_old_young.X.copy()
ife_old_young.raw = sc.pp.log1p(ife_old_young, copy = True)

### Data normalization and scale

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(ife_old_young)
# Logarithmize the data
sc.pp.log1p(ife_old_young)

In [ ]:
#Highly variable selection
sc.pp.highly_variable_genes(ife_old_young, n_top_genes=2000, batch_key='cell_age')

In [ ]:
#Cell cycle scoring
cell_cycle_genes = [x.strip().title() for x in open('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/regev_lab_cell_cycle_genes.txt')]
# Separate dataset for S-phase or G2M-phase associated genes
s_genes = [x for x in cell_cycle_genes[:43]]
g2m_genes = [x for x in cell_cycle_genes[43:]]
sc.tl.score_genes_cell_cycle(ife_old_young, s_genes = s_genes, g2m_genes=g2m_genes)

In [ ]:
ife_old_young.obs['CC_Difference'] = ife_old_young.obs[str('S_score')] - ife_old_young.obs[str('G2M_score')]

In [ ]:
ife_old_young.obs

In [ ]:
sc.pl.highly_variable_genes(ife_old_young)

In [ ]:
ife_old_young.raw = ife_old_young

In [ ]:
ife_old_young.obs

In [ ]:
sc.pp.regress_out(ife_old_young, ["pct_counts_mt", 'pct_counts_ribo'])

In [ ]:
sc.pp.scale(ife_old_young, max_value=10)

### PCA, UMAP and Clustering

In [ ]:
#PCA
sc.tl.pca(ife_old_young)

In [ ]:
sc.pl.pca_variance_ratio(ife_old_young, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    ife_old_young,
    color=["cell_age", "cell_age", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

In [ ]:
#Nearest neighbor
sc.pp.neighbors(ife_old_young, n_neighbors=30)
#Umap
sc.tl.umap(ife_old_young, min_dist=1, spread=2)

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster, especially for larger datasets
sc.tl.leiden(ife_old_young, flavor="igraph", n_iterations=-1, resolution = 0.2)

In [ ]:
# Read directly to just plot
ife_old_young = sc.read_h5ad('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/ife_old_young.h5ad')

In [ ]:
sc.pl.umap(ife_old_young, color=["leiden"])

In [ ]:
sc.pl.umap(
    ife_old_young,
    color=["leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
)

In [ ]:
sc.pl.umap(
    ife_old_young,
    color=["leiden", "phase", "pct_counts_mt", "log1p_n_genes_by_counts", 'pct_counts_ribo', 'pct_counts_diss'],
    wspace=0.5,
    ncols=3,
)

In [ ]:
fig, axes = plt.subplots(ncols = 2, nrows=1, figsize = (10, 4))
size = 15
ax = axes[0]
wspace = 1,
gene = 'Irf6'
sc.pl.umap(ife_old_young, show = False, ax = ax, size = size)
sc.pl.umap(ife_old_young[ife_old_young.obs['cell_age']=='ife_old'], color= gene,  size = size, title= 'Old peroxisomes', wspace = wspace, ax = ax, show = False)
ax = axes[1]
sc.pl.umap(ife_old_young, show = False, ax = ax, size = size)
sc.pl.umap(ife_old_young[ife_old_young.obs['cell_age']=='ife_young'], color= gene, size = size, title= 'Young peroxisomes', wspace = wspace, ax = ax, show = False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/ife_umap_orig.tif', format='tiff',bbox_inches='tight', dpi = 300)

### Cluster annotation

In [ ]:
sc.pl.umap(
    ife_old_young,
    color=["Ccnb1", "Cdk1", "Fth1", "Apoe", "Krtdap", "Sostdc1", "Ivl", "Krt10", "Krt14", 'leiden'],
    wspace=0.5,
    ncols=3,
    cmap= 'viridis'
)

In [ ]:
#Identification of clusters based on known markers

marker_genes = {
    'IFE_Basal':['Krt14', 'Fth1'],
    'IFE_Basal_C': ['Cdk1', 'Ccnb1'], 
    'Suprabasal': ['Krt10', 'Krtdap'],
    'Infundibulum': ['Sostdc1']
}
sc.pl.dotplot(ife_old_young, marker_genes, 'leiden', dendrogram=True)

In [ ]:
#Create a dictionary to map clusters to annotation labels

Cluster2annotation = {
    '1': 'Basal', 
    '3': 'Infundibulum', 
    '2': 'Suprbasal',
    '4': 'IFE_Cycling'
}
#Add annotations to obs
ife_old_young.obs["cell type"] = ife_old_young.obs["leiden"].map(Cluster2annotation).astype("category")
sc.pl.dotplot(ife_old_young, marker_genes, 'cell type', dendrogram=True)

In [ ]:
ax = sc.pl.umap(
    ife_old_young,
    color="cell type",
    legend_loc="on data",
    frameon=False,
    legend_fontsize=5, size=10,
    show = False, na_in_legend = False)
sc.pl.umap(ife_old_young, color="cell type",
    frameon=False,
    legend_fontsize=5, size = 10, ax = ax, na_in_legend = False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/ife_umap.tif', format='tiff',bbox_inches='tight', dpi = 300)
plt.show()

In [ ]:
ax = sc.pl.umap(
    ife_old_young,
    color="cell type",
    legend_loc="on data",
    frameon=False,
    legend_fontsize=5, 
    size=10,
    show=False, na_in_legend=False
)

sc.pl.umap(
    ife_old_young,
    color="cell type",
    frameon=False,
    legend_fontsize=5, 
    size=10,  # Ensure size is not zero
    ax=ax, na_in_legend=False,
    show=False  # Keep it False so plt.savefig works
)

plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/ife_umap.tif', 
            format='tiff', bbox_inches='tight', dpi=300)

plt.show()  # Optional but useful if running interactively


In [ ]:
ife_old_young.write(results_file)

In [ ]:
sc.pl.umap(
    ife_old_young,
    color=["Ivl", "Krt10", "Krt14"],
    wspace=0.5,
    ncols=3,
)

### DE expressed genes

In [ ]:
Basal = ife_old_young[ife_old_young.obs.leiden=='1'].copy()
Basal.X = Basal.layers['counts'].toarray().copy()

In [ ]:
SBasal = ife_old_young[ife_old_young.obs.leiden=='2'].copy()
SBasal.X = Basal.layers['counts'].toarray().copy()

In [ ]:
sc.pl.umap(Basal, color='Irf6')

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(Basal, groupby="cell_age", method="wilcoxon")

In [ ]:
ife_DE = sc.get.rank_genes_groups_df(Basal, group="ife_old")
ife_DE.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/old_pero_basal_genes.csv', index=False)

In [ ]:
top_15_DE_genes

In [ ]:
#Volcano Plot
DE_genes = ife_DE
#Add -log10(pval) column
DE_genes['-log10(pvals_adj)'] = -np.log10(DE_genes['pvals_adj'])
#logfoldchange threshold
logFC_threshold = 10
DE_genes = DE_genes[abs(DE_genes['logfoldchanges']) <= logFC_threshold]
# Define thresholds for significance
log2FoldChange_threshold = 1 
pval_threshold = 0.05 
s = 10
# Determine significance
DE_genes['significant'] = (abs(DE_genes['logfoldchanges']) > log2FoldChange_threshold) & (DE_genes['pvals_adj'] < pval_threshold)
# Get top 15 significant genes sorted by p-value (or you could sort by log2 fold change if desired)
top_15_DE_genes = DE_genes[DE_genes['significant']].nsmallest(25, 'pvals_adj')
# Plotting
plt.figure(figsize=(5, 4))
# Scatter plot for non-significant genes
plt.scatter(DE_genes['logfoldchanges'], DE_genes['-log10(pvals_adj)'], c='grey', alpha=0.7, s=s)
# Scatter plot for significant genes (color coding by sign of logfoldchange)
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), '-log10(pvals_adj)'], 
            c='red', alpha=0.7, s=s, label='Upregulated')
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), '-log10(pvals_adj)'], 
            c='blue', alpha=0.7, s=s, label='Downregulated')
# Label significant genes
# Label the top 15 significant genes
for i, row in top_15_DE_genes.iterrows():
    plt.text(row['logfoldchanges'], row['-log10(pvals_adj)'], row['names'], fontsize=8, ha='right')
# Adding labels and title
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 p-value')
plt.title('IFE Old vs IFE young DEG')
plt.axhline(-np.log10(pval_threshold), color='green', linestyle='--')  # Add a horizontal line for p-value threshold
plt.axvline(log2FoldChange_threshold, color='green', linestyle='--')   # Add vertical lines for log2 fold change threshold
plt.axvline(-log2FoldChange_threshold, color='green', linestyle='--')
plt.grid(False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/IFE_DEG.tif', format='tiff', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
IFE_DE_Up = ife_DE.sort_values('logfoldchanges', ascending = False).query('pvals_adj<0.05').head(50)
#Bulge_DE_Up.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Bulge_DE_Up.csv', index=False)
IFE_DE_Up

In [ ]:
IFE_DE_Down = ife_DE.sort_values('logfoldchanges', ascending = True).query('pvals_adj<0.05').head(50)
#Bulge_DE_Up.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Bulge_DE_Up.csv', index=False)
IFE_DE_Down

In [ ]:
genes_up = ['Hoxc8', 'Tgm3', 'Hoxa7', 'Krtdap', 'Fam71a', 'Timp3', 'Hoxa5', 'Zbtb7c', 'Calm5', 'Nuak1']
genes_down = ['B930036N10Rik', '2900060B14Rik', 'Gm11867', 'Gm49890', 'Mmp10', 'Ccdc80', 'Snhg9', 'Ngf', 'Aff2', 'Ltbp1']

fig, axs = plt.subplots(2, 1, figsize=(8, 5))
#sc.pl.rank_genes_groups_dotplot(Bulge, groups = ['hf_old'], groupby = 'cell_age', dendrogram = False, var_names= genes_up, ax=axs [0], show=False)
sc.pl.rank_genes_groups_dotplot(Basal, groups = ['ife_old'], groupby = 'cell_age', dendrogram = False, values_to_plot='logfoldchanges', var_names= genes_up, ax=axs [0], show=False)
#sc.pl.rank_genes_groups_dotplot(Bulge, groups = ['hf_young'], groupby = 'cell_age', dendrogram = False, var_names= genes_down, ax=axs [2], show=False)
sc.pl.rank_genes_groups_dotplot(Basal, groups = ['ife_young'], groupby = 'cell_age', dendrogram = False, values_to_plot='logfoldchanges', var_names=genes_down, ax=axs [1], show=False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/Basal_DEG_dotplot.tif', format='tiff', bbox_inches='tight', dpi=300)

In [ ]:
#Volcano Plot
DE_genes = old_pero_basal_genes
#Add -log10(pval) column
DE_genes['-log10(pval)'] = -np.log10(DE_genes['pvals'])
#logfoldchange threshold
logFC_threshold = 10
DE_genes = DE_genes[abs(DE_genes['logfoldchanges']) <= logFC_threshold]
# Define thresholds for significance
log2FoldChange_threshold = 1  
pval_threshold = 0.05 
s = 10
# Determine significance
DE_genes['significant'] = (abs(DE_genes['logfoldchanges']) > log2FoldChange_threshold) & (DE_genes['pvals'] < pval_threshold)
# Get top 15 significant genes sorted by p-value (or you could sort by log2 fold change if desired)
top_15_DE_genes = DE_genes[DE_genes['significant']].nsmallest(50, 'pvals')
# Plotting
plt.figure(figsize=(5, 4))
# Scatter plot for non-significant genes
plt.scatter(DE_genes['logfoldchanges'], DE_genes['-log10(pval)'], c='grey', alpha=0.7, s=s)
# Scatter plot for significant genes (color coding by sign of logfoldchange)
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), '-log10(pval)'], 
            c='red', alpha=0.7, s=s, label='Upregulated')
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), '-log10(pval)'], 
            c='blue', alpha=0.7, s=s, label='Downregulated')
# Label significant genes
# Label the top 15 significant genes
for i, row in top_15_DE_genes.iterrows():
    plt.text(row['logfoldchanges'], row['-log10(pval)'], row['names'], fontsize=8, ha='right')
# Adding labels and title
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 p-value')
plt.title('Subcluster 0 Old DEGs')
plt.axhline(-np.log10(pval_threshold), color='green', linestyle='--')  # Add a horizontal line for p-value threshold
plt.axvline(log2FoldChange_threshold, color='green', linestyle='--')   # Add vertical lines for log2 fold change threshold
plt.axvline(-log2FoldChange_threshold, color='green', linestyle='--')
plt.grid(False)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/Subcluster_0_DEG.tif', format='tiff', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
ife_old_young.obs

In [ ]:
Basal_C = ife_old_young[ife_old_young.obs.leiden=='4'].copy()
Basal_C.X = Basal_C.layers['counts'].toarray().copy()

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(Basal_C, groupby="cell_age", method="wilcoxon")

In [ ]:
old_pero_basal_c_genes = sc.get.rank_genes_groups_df(Basal_C, group="ife_old")
old_pero_basal_c_genes.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/old_pero_basal_c_genes.csv', index=False)

In [ ]:
top_15_DE_genes

In [ ]:
sc.pl.violin(ife_old_young, keys = ['Krt77','Pex6', 'Hoxc8'], groupby = 'cell type')
sc.pl.violin(ife_old_young, keys = ['Ccdc80','Ripor1', 'Igfbp3'], groupby = 'cell type')

In [ ]:
#Volcano Plot
DE_genes = old_pero_basal_c_genes
#Add -log10(pval) column
DE_genes['-log10(pval)'] = -np.log10(DE_genes['pvals'])
#logfoldchange threshold
logFC_threshold = 10
DE_genes = DE_genes[abs(DE_genes['logfoldchanges']) <= logFC_threshold]
# Define thresholds for significance
log2FoldChange_threshold = 1  
pval_threshold = 0.05 
s = 10
# Determine significance
DE_genes['significant'] = (abs(DE_genes['logfoldchanges']) > log2FoldChange_threshold) & (DE_genes['pvals'] < pval_threshold)
# Get top 15 significant genes sorted by p-value (or you could sort by log2 fold change if desired)
top_15_DE_genes = DE_genes[DE_genes['significant']].nsmallest(50, 'pvals')
# Plotting
plt.figure(figsize=(5, 4))
# Scatter plot for non-significant genes
plt.scatter(DE_genes['logfoldchanges'], DE_genes['-log10(pval)'], c='grey', alpha=0.7, s=s)
# Scatter plot for significant genes (color coding by sign of logfoldchange)
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), '-log10(pval)'], 
            c='red', alpha=0.7, s=s, label='Upregulated')
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), '-log10(pval)'], 
            c='blue', alpha=0.7, s=s, label='Downregulated')
# Label significant genes
# Label the top 15 significant genes
for i, row in top_15_DE_genes.iterrows():
    plt.text(row['logfoldchanges'], row['-log10(pval)'], row['names'], fontsize=8, ha='right')
# Adding labels and title
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 p-value')
plt.title('Subcluster 0 Old DEGs')
plt.axhline(-np.log10(pval_threshold), color='green', linestyle='--')  # Add a horizontal line for p-value threshold
plt.axvline(log2FoldChange_threshold, color='green', linestyle='--')   # Add vertical lines for log2 fold change threshold
plt.axvline(-log2FoldChange_threshold, color='green', linestyle='--')
plt.grid(False)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/Subcluster_0_DEG.tif', format='tiff', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
ife_old_young.obs['scluster_age'] = ife_old_young.obs['leiden'].astype(str) + '_' + ife_old_young.obs['cell_age'].astype(str)

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(ife_old_young, groupby="scluster_age", method="wilcoxon", groups= ['0_ife_old', '2_ife_old'])

In [ ]:
old_pero_c0_genes = sc.get.rank_genes_groups_df(ife_old_young, group="0_ife_old")
old_pero_c0_genes.to_csv('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/old_pero_c0_DEG.csv', index=False)

In [ ]:
#Volcano Plot
DE_genes = old_pero_c0_genes
#Add -log10(pval) column
DE_genes['-log10(pval)'] = -np.log10(DE_genes['pvals'])
#logfoldchange threshold
logFC_threshold = 10
DE_genes = DE_genes[abs(DE_genes['logfoldchanges']) <= logFC_threshold]
# Define thresholds for significance
log2FoldChange_threshold = 1  
pval_threshold = 0.05 
s = 10
# Determine significance
DE_genes['significant'] = (abs(DE_genes['logfoldchanges']) > log2FoldChange_threshold) & (DE_genes['pvals'] < pval_threshold)
# Get top 15 significant genes sorted by p-value (or you could sort by log2 fold change if desired)
top_15_DE_genes = DE_genes[DE_genes['significant']].nsmallest(50, 'pvals')
# Plotting
plt.figure(figsize=(5, 4))
# Scatter plot for non-significant genes
plt.scatter(DE_genes['logfoldchanges'], DE_genes['-log10(pval)'], c='grey', alpha=0.7, s=s)
# Scatter plot for significant genes (color coding by sign of logfoldchange)
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] > 0), '-log10(pval)'], 
            c='red', alpha=0.7, s=s, label='Upregulated')
plt.scatter(DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), 'logfoldchanges'], 
            DE_genes.loc[(DE_genes['significant']) & (DE_genes['logfoldchanges'] < 0), '-log10(pval)'], 
            c='blue', alpha=0.7, s=s, label='Downregulated')
# Label significant genes
# Label the top 15 significant genes
for i, row in top_15_DE_genes.iterrows():
    plt.text(row['logfoldchanges'], row['-log10(pval)'], row['names'], fontsize=8, ha='right')
# Adding labels and title
plt.xlabel('Log2 Fold Change')
plt.ylabel('-Log10 p-value')
plt.title('Subcluster 0 Old DEGs')
plt.axhline(-np.log10(pval_threshold), color='green', linestyle='--')  # Add a horizontal line for p-value threshold
plt.axvline(log2FoldChange_threshold, color='green', linestyle='--')   # Add vertical lines for log2 fold change threshold
plt.axvline(-log2FoldChange_threshold, color='green', linestyle='--')
plt.grid(False)
#plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/Subcluster_0_DEG.tif', format='tiff', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
top_15_DE_genes

In [ ]:
ife_old_young.write(results_file)

# EPI_OLD_YOUNG

## Read data 

In [ ]:
results_file = '/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/epi_old_young.h5ad'

In [ ]:
#Read data
ife_old = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_056/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
ife_young = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_057/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
hf_old = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_059/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
hf_young = sc.read_10x_mtx('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/files/P.Katajisto_23_01/10X_23_060/outs/filtered_feature_bc_matrix')
var_names = 'gene_symbols'
cache=True
#Merge data
epi_old_young = sc.concat([ife_old, ife_young, hf_old, hf_young], axis=0, label='cell_age', keys= ['ife_old', 'ife_young', 'hf_old', 'hf_young'], index_unique='_', join='outer')
epi_old_young.obs_names_make_unique()

## Preprocessing

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
epi_old_young.var["mt"] = epi_old_young.var_names.str.startswith("mt-")
# ribosomal genes
epi_old_young.var["ribo"] = epi_old_young.var_names.str.startswith(("Rps", "Rpl"))
# dissociation genes
diss_genes = ["Atf3", "Btg2", "Dusp1", "Egr1", "Fos", "Fosb", "Ier2", "Ifrd1", "Junb", "Nr4a1", "Zfp36"]
epi_old_young.var['diss'] = epi_old_young.var_names.isin(diss_genes)
# calculate qc metrics
sc.pp.calculate_qc_metrics(epi_old_young, qc_vars=["mt", "ribo", "diss"], inplace=True, log1p=True)

In [ ]:
#Plot qc metrics_violin
sc.pl.violin(epi_old_young,["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_diss"], jitter=0.4, multi_panel=True)

In [ ]:
#Plot qc metrics_scatter
sc.pl.scatter(epi_old_young, 'total_counts', 'n_genes_by_counts', color='pct_counts_mt')

In [ ]:
#Filter cells
sc.pp.filter_cells(epi_old_young, min_genes=100)
sc.pp.filter_genes(epi_old_young, min_cells=3)
epi_old_young = epi_old_young[epi_old_young.obs.n_genes_by_counts < 6500, :]
epi_old_young = epi_old_young[epi_old_young.obs.pct_counts_mt < 20, :].copy()
epi_old_young = epi_old_young[epi_old_young.obs.pct_counts_ribo < 25, :].copy()

In [ ]:
#Doublets detection
sc.pp.scrublet(epi_old_young)

In [ ]:
epi_old_young.obs

In [ ]:
# Saving count data
epi_old_young.layers["counts"] = epi_old_young.X.copy()

In [ ]:
epi_old_young.raw = sc.pp.log1p(epi_old_young, copy = True)

## Data normalization and scale

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(epi_old_young)
# Logarithmize the data
sc.pp.log1p(epi_old_young)

In [ ]:
#Highly variable selection
sc.pp.highly_variable_genes(epi_old_young, n_top_genes=2000, batch_key='cell_age')

In [ ]:
#Cell cycle scoring
cell_cycle_genes = [x.strip().title() for x in open('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/regev_lab_cell_cycle_genes.txt')]
# Separate dataset for S-phase or G2M-phase associated genes
s_genes = [x for x in cell_cycle_genes[:43]]
g2m_genes = [x for x in cell_cycle_genes[43:]]
sc.tl.score_genes_cell_cycle(epi_old_young, s_genes = s_genes, g2m_genes=g2m_genes)

In [ ]:
epi_old_young.obs['CC_Difference'] = epi_old_young.obs[str('S_score')] - epi_old_young.obs[str('G2M_score')]

In [ ]:
sc.pl.highly_variable_genes(epi_old_young)

In [ ]:
sc.pp.regress_out(epi_old_young, ["pct_counts_mt", 'pct_counts_ribo', 'pct_counts_diss', 'CC_Difference'])

In [ ]:
sc.pp.scale(epi_old_young, max_value=10)

## PCA, UMAP and clustering

In [ ]:
#PCA
sc.tl.pca(epi_old_young)

In [ ]:
sc.pl.pca_variance_ratio(epi_old_young, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    epi_old_young,
    color=["cell_age", "cell_age", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

In [ ]:
#Nearest neighbor
sc.pp.neighbors(epi_old_young, n_neighbors=30)
#Umap
sc.tl.umap(epi_old_young, min_dist=1, spread=2)

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster, especially for larger datasets
sc.tl.leiden(epi_old_young, flavor="igraph", n_iterations=-1, resolution = 0.5)

In [ ]:
# Read directly to just plot
epi_old_young = sc.read_h5ad('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/epi_old_young.h5ad')

In [ ]:
plt.style.use('dark_background')
sc.pl.umap(epi_old_young, color=["leiden"], frameon=False)

In [ ]:
plt.style.use('dark_background')
sc.pl.umap(epi_old_young, color=["cell_age"], show=False, frameon=False)
plt.savefig('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/Scanpy_10X Old Young Analysis/Figures/epi_old_young_cell_age_darl.tif', format='tiff', bbox_inches='tight', dpi=300 )

In [ ]:
sc.pl.umap(
    epi_old_young,
    color=["leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
)

In [ ]:
sc.pl.umap(
    epi_old_young,
    color=["leiden", "phase", "pct_counts_mt", "log1p_n_genes_by_counts", 'pct_counts_ribo', 'pct_counts_diss'],
    wspace=0.5,
    ncols=3,
)

In [ ]:
sc.pl.umap(
    epi_old_young,
    color=["Ccnb1", "Cdk1", "Fth1", "Krt79", "Krtdap", "Sostdc1", "Cd34", "Krt10", "Krt14", 'Lgr5', 'Lgr6', 'Lrig1', 'Cidea', 'Mgst1'],
    wspace=0.5,
    ncols=3,
)

In [ ]:
epi_old_young.write(results_file)

# MKA data

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
sc.settings.set_figure_params(dpi=150, facecolor="white")

In [ ]:
adata = sc.read_h5ad('/Users/agustin.sola.carvajal/Desktop/DataDelivery_2024-02-12_11-10-20_ngisthlm00563/MKA data/Anagen_10x_CSC2020_Old_MetadataCombined_Permanent_epidermis.h5ad')

In [ ]:
adata.obs